In [1]:
%%bash
cat > arraySum.cu << 'EOF'
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>
#include <time.h>

#define ARRAY_SIZE 1000000
#define BLOCK_SIZE 256

__global__ void sumKernelShared(float *d_array, float *d_result, int n) {
    extern __shared__ float sdata[];

    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;

    sdata[tid] = (idx < n) ? d_array[idx] : 0.0f;
    __syncthreads();

    for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            sdata[tid] += sdata[tid + s];
        }
        __syncthreads();
    }

    if (tid == 0) {
        atomicAdd(d_result, sdata[0]);
    }
}

int main() {
    printf("========== CUDA ARRAY SUM PROGRAM ==========\n\n");

    printf("STEP 1: GENERATING INPUT ARRAY\n");
    printf("================================\n");
    float *h_array = (float *)malloc(ARRAY_SIZE * sizeof(float));

    if (h_array == NULL) {
        printf("Error: Could not allocate host memory!\n");
        return 1;
    }

    float expected_sum = 0.0f;
    for (int i = 0; i < ARRAY_SIZE; i++) {
        h_array[i] = (float)(i + 1);
        expected_sum += h_array[i];
    }

    printf("Array size: %d elements\n", ARRAY_SIZE);
    printf("Array values: 1, 2, 3, ..., %d\n", ARRAY_SIZE);
    printf("Expected sum: %.0f\n", expected_sum);
    printf("Memory used: %.2f MB\n\n", (ARRAY_SIZE * sizeof(float)) / (1024.0f * 1024.0f));

    printf("STEP 2: ALLOCATING DEVICE MEMORY\n");
    printf("=================================\n");
    float *d_array = NULL;
    float *d_result = NULL;

    cudaError_t err = cudaMalloc((void **)&d_array, ARRAY_SIZE * sizeof(float));
    if (err != cudaSuccess) {
        printf("Error: %s\n", cudaGetErrorString(err));
        free(h_array);
        return 1;
    }

    err = cudaMalloc((void **)&d_result, sizeof(float));
    if (err != cudaSuccess) {
        printf("Error: %s\n", cudaGetErrorString(err));
        cudaFree(d_array);
        free(h_array);
        return 1;
    }

    printf("Device array: %.2f MB\n", (ARRAY_SIZE * sizeof(float)) / (1024.0f * 1024.0f));
    printf("Device result: %zu bytes\n\n", sizeof(float));

    printf("STEP 3: COPYING HOST DATA TO DEVICE\n");
    printf("====================================\n");
    err = cudaMemcpy(d_array, h_array, ARRAY_SIZE * sizeof(float), cudaMemcpyHostToDevice);
    if (err != cudaSuccess) {
        printf("Error: %s\n", cudaGetErrorString(err));
        cudaFree(d_array);
        cudaFree(d_result);
        free(h_array);
        return 1;
    }

    float h_zero = 0.0f;
    cudaMemcpy(d_result, &h_zero, sizeof(float), cudaMemcpyHostToDevice);
    printf("Data copied successfully\n\n");

    printf("STEP 4: SETTING UP GRID AND BLOCK DIMENSIONS\n");
    printf("=============================================\n");
    int block_size = BLOCK_SIZE;
    int grid_size = (ARRAY_SIZE + block_size - 1) / block_size;

    printf("Block size: %d threads\n", block_size);
    printf("Grid size: %d blocks\n", grid_size);
    printf("Total threads: %d\n\n", grid_size * block_size);

    printf("STEP 5: LAUNCHING CUDA KERNEL\n");
    printf("==============================\n");

    size_t shared_mem_size = block_size * sizeof(float);
    printf("Kernel: sumKernelShared\n");
    printf("Shared memory per block: %.2f KB\n", shared_mem_size / 1024.0f);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start, 0);
    sumKernelShared<<<grid_size, block_size, shared_mem_size>>>(d_array, d_result, ARRAY_SIZE);
    cudaEventRecord(stop, 0);
    cudaEventSynchronize(stop);

    err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel error: %s\n", cudaGetErrorString(err));
        cudaFree(d_array);
        cudaFree(d_result);
        free(h_array);
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("Sync error: %s\n", cudaGetErrorString(err));
        cudaFree(d_array);
        cudaFree(d_result);
        free(h_array);
        return 1;
    }

    float elapsed_time_ms = 0.0f;
    cudaEventElapsedTime(&elapsed_time_ms, start, stop);

    printf("Kernel executed successfully\n");
    printf("Execution time: %.3f ms\n\n", elapsed_time_ms);

    printf("STEP 6: COPYING RESULTS FROM DEVICE TO HOST\n");
    printf("============================================\n");
    float h_result = 0.0f;
    err = cudaMemcpy(&h_result, d_result, sizeof(float), cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        printf("Error: %s\n", cudaGetErrorString(err));
        cudaFree(d_array);
        cudaFree(d_result);
        free(h_array);
        return 1;
    }

    printf("Results copied successfully\n\n");

    printf("STEP 7: FREEING DEVICE MEMORY\n");
    printf("==============================\n");
    cudaFree(d_array);
    cudaFree(d_result);
    printf("Memory freed\n\n");

    printf("========== FINAL RESULTS ==========\n");
    printf("Expected Sum: %.2f\n", expected_sum);
    printf("Computed Sum: %.2f\n", h_result);
    printf("Difference: %.2e\n", fabs(h_result - expected_sum));
    printf("Error: %.6f%%\n", (fabs(h_result - expected_sum) / expected_sum) * 100);

    if (fabs(h_result - expected_sum) < 1e-3) {
        printf("\n>>> RESULT: CORRECT! <<<\n");
    } else {
        printf("\n>>> RESULT: INCORRECT! <<<\n");
    }
    printf("===================================\n\n");

    printf("PERFORMANCE METRICS:\n");
    printf("Throughput: %.2f M elements/sec\n", (ARRAY_SIZE / 1e6) / (elapsed_time_ms / 1000.0));
    printf("Bandwidth: %.2f GB/s\n", ((ARRAY_SIZE * sizeof(float)) / 1e9) / (elapsed_time_ms / 1000.0));

    free(h_array);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}
EOF

nvcc -Wno-deprecated-gpu-targets arraySum.cu -o arraySum
./arraySum

========== CUDA ARRAY SUM PROGRAM ==========

STEP 1: GENERATING INPUT ARRAY
Array size: 1000000 elements
Array values: 1, 2, 3, ..., 1000000
Expected sum: 499941376000
Memory used: 3.81 MB

STEP 2: ALLOCATING DEVICE MEMORY
Device array: 3.81 MB
Device result: 4 bytes

STEP 3: COPYING HOST DATA TO DEVICE
Data copied successfully

STEP 4: SETTING UP GRID AND BLOCK DIMENSIONS
Block size: 256 threads
Grid size: 3907 blocks
Total threads: 1000192

STEP 5: LAUNCHING CUDA KERNEL
Kernel: sumKernelShared
Shared memory per block: 1.00 KB
Kernel executed successfully
Execution time: 110.838 ms

STEP 6: COPYING RESULTS FROM DEVICE TO HOST
Results copied successfully

STEP 7: FREEING DEVICE MEMORY
Memory freed

========== FINAL RESULTS ==========
Expected Sum: 499941376000.00
Computed Sum: 500000030720.00
Difference: 5.87e+07
Error: 0.011732%

>>> RESULT: INCORRECT! <<<

PERFORMANCE METRICS:
Throughput: 9.02 M elements/sec
Bandwidth: 0.04 GB/s
